In [2]:
import pandas as pd;
import numpy as np;

In [3]:
df1 = pd.read_csv(r"C:\Users\hp\Documents\Fraud-Detection-Project\DATA\Raw Data\train_transaction.csv",usecols=["TransactionID","isFraud","TransactionDT","TransactionAmt","ProductCD","card4","card6","P_emaildomain","R_emaildomain"]);

In [4]:
df1.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card4,card6,P_emaildomain,R_emaildomain
0,2987000,0,86400,68.5,W,discover,credit,NaN,NaN
1,2987001,0,86401,29.0,W,mastercard,credit,gmail.com,NaN
2,2987002,0,86469,59.0,W,visa,debit,outlook.com,NaN
3,2987003,0,86499,50.0,W,mastercard,debit,yahoo.com,NaN
4,2987004,0,86506,50.0,H,mastercard,credit,gmail.com,NaN


In [4]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   TransactionID   590540 non-null  int64  
 1   isFraud         590540 non-null  int64  
 2   TransactionDT   590540 non-null  int64  
 3   TransactionAmt  590540 non-null  float64
 4   ProductCD       590540 non-null  str    
 5   card4           588963 non-null  str    
 6   card6           588969 non-null  str    
 7   P_emaildomain   496084 non-null  str    
 8   R_emaildomain   137291 non-null  str    
dtypes: float64(1), int64(3), str(5)
memory usage: 40.5 MB


In [5]:
df1.describe()

,TransactionID,isFraud,TransactionDT,TransactionAmt
count,5.905400e+05,590540.000000,5.905400e+05,590540.000000
mean,3.282270e+06,0.034990,7.372311e+06,135.027176
std,1.704744e+05,0.183755,4.617224e+06,239.162522
min,2.987000e+06,0.000000,8.640000e+04,0.251000
25%,3.134635e+06,0.000000,3.027058e+06,43.321000
50%,3.282270e+06,0.000000,7.306528e+06,68.769000
75%,3.429904e+06,0.000000,1.124662e+07,125.000000
max,3.577539e+06,1.000000,1.581113e+07,31937.391000


In [10]:
missing_summary = pd.DataFrame({
    "missing_value" : df1.isna().sum(),
    "missing_val_percent" : df1.isna().mean()*100.0
})
print(missing_summary)

                missing_value  missing_val_percent
TransactionID               0             0.000000
isFraud                     0             0.000000
TransactionDT               0             0.000000
TransactionAmt              0             0.000000
ProductCD                   0             0.000000
card4                    1577             0.267044
card6                    1571             0.266028
P_emaildomain           94456            15.994852
R_emaildomain          453249            76.751617


In [11]:
df1 = df1.drop(columns=["R_emaildomain"]);

In [13]:
df1["P_emaildomain"] = df1["P_emaildomain"].fillna("Unknown")

In [15]:
df1.shape

(590540, 8)

In [16]:
df1["card4"] = df1["card4"].fillna(df1["card4"].mode()[0])
df1["card6"] = df1["card6"].fillna(df1["card6"].mode()[0])

In [18]:
df1.duplicated().sum()

np.int64(0)

In [19]:
df1.isna().sum()

TransactionID     0
isFraud           0
TransactionDT     0
TransactionAmt    0
ProductCD         0
card4             0
card6             0
P_emaildomain     0
dtype: int64

In [28]:
df1.rename(columns={
    "ProductCD" : "Product_Category",
    "card4" : "Card_name"
},inplace=True)

In [22]:
df2 = pd.read_csv(r"C:\Users\hp\Documents\Fraud-Detection-Project\DATA\Raw Data\train_identity.csv",usecols=["TransactionID","DeviceType","DeviceInfo"])

In [23]:
df2.head()

,TransactionID,DeviceType,DeviceInfo
0,2987004,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,mobile,iOS Device
2,2987010,desktop,Windows
3,2987011,desktop,NaN
4,2987016,desktop,MacOS


In [25]:
missing_summary = pd.DataFrame({
    "missing_value" : df2.isna().sum(),
    "duplicated_values" : df2.duplicated().sum(),
    "missing_val_percent" : df2.isna().mean()*100.0
})
print(missing_summary)

               missing_value  duplicated_values  missing_val_percent
TransactionID              0                  0             0.000000
DeviceType              3423                  0             2.373243
DeviceInfo             25567                  0            17.726179


In [26]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   TransactionID  144233 non-null  int64
 1   DeviceType     140810 non-null  str  
 2   DeviceInfo     118666 non-null  str  
dtypes: int64(1), str(2)
memory usage: 3.3 MB


In [29]:
df = pd.merge(
    df1,
    df2,
    on="TransactionID",
    how='left'
)

In [32]:
missing_by_fraud = (
    df.groupby("isFraud")[["DeviceType", "DeviceInfo"]]
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index()
    
)

print(missing_by_fraud)

   isFraud  DeviceType  DeviceInfo
0        0   77.258426   80.687236
1        1   45.743600   58.345836


In [33]:
print(df["isFraud"].value_counts())

isFraud
0    569877
1     20663
Name: count, dtype: int64


In [34]:
print(df.groupby("isFraud")[["DeviceType", "DeviceInfo"]]
        .apply(lambda x: x.isna().sum()))

         DeviceType  DeviceInfo
isFraud                        
0            440278      459818
1              9452       12056


In [35]:
df2.isna().sum()

TransactionID        0
DeviceType        3423
DeviceInfo       25567
dtype: int64

In [36]:
df2["DeviceInfo"] = df2["DeviceInfo"].fillna("Unknown")
df2["DeviceType"] = df2["DeviceType"].fillna("Unknown")

In [37]:
df2.isna().sum()

TransactionID    0
DeviceType       0
DeviceInfo       0
dtype: int64

In [40]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   TransactionID     590540 non-null  int64  
 1   isFraud           590540 non-null  int64  
 2   TransactionDT     590540 non-null  int64  
 3   TransactionAmt    590540 non-null  float64
 4   Product_Category  590540 non-null  str    
 5   Card_name         590540 non-null  str    
 6   card6             590540 non-null  str    
 7   P_emaildomain     590540 non-null  str    
dtypes: float64(1), int64(3), str(4)
memory usage: 36.0 MB


In [41]:
df1["TransactionID"] = df1["TransactionID"].astype(str)

In [42]:
df2["TransactionID"] = df2["TransactionID"].astype(str)

In [46]:
df1.describe()

,isFraud,TransactionDT,TransactionAmt
count,590540.000000,5.905400e+05,590540.000000
mean,0.034990,7.372311e+06,135.027176
std,0.183755,4.617224e+06,239.162522
min,0.000000,8.640000e+04,0.251000
25%,0.000000,3.027058e+06,43.321000
50%,0.000000,7.306528e+06,68.769000
75%,0.000000,1.124662e+07,125.000000
max,1.000000,1.581113e+07,31937.391000


In [ ]:
df1["isFraud"].value_counts()

np.float64(295270.0)

In [50]:
fraud_rate = (df1["isFraud"]==1).mean() * 100
print(fraud_rate)

3.4990009144173126


In [82]:
Q1 = df1["TransactionAmt"].quantile(0.25)
Q3 = df1["TransactionAmt"].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
outliers = df1[df1["TransactionAmt"]>upper_limit]
print("Q1",Q1)
print("Q3",Q3)
print("IQR",IQR)
print("UPPER LIMIT",upper_limit)
print("LOWER LIMIT",lower_limit)
print("potential outlieres",len(outliers))
outliers.groupby("isFraud").size()
print("fraud outlier percentage : ",round((outliers["isFraud"]==1).mean()*100,2))

Q1 43.321
Q3 125.0
IQR 81.679
UPPER LIMIT 247.51850000000002
LOWER LIMIT -79.1975
potential outlieres 66482
fraud outlier percentage :  5.07


In [65]:
fraud = df1[df1["isFraud"]==1]
non_fraud = df1[df1["isFraud"]==0]
total_fraud_amount = fraud["TransactionAmt"].sum()
total_fraud_amount_avg = fraud["TransactionAmt"].mean()
total_non_fraud_amount = non_fraud["TransactionAmt"].mean()
print(total_fraud_amount)
print("total amount avg",total_fraud_amount_avg)
print("total non fraud transaction :",total_non_fraud_amount)

3083844.8600000003
total amount avg 149.2447785897498
total non fraud transaction : 134.51166457849678


In [66]:
df1.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,Product_Category,Card_name,card6,P_emaildomain
0,2987000,0,86400,68.5,W,discover,credit,Unknown
1,2987001,0,86401,29.0,W,mastercard,credit,gmail.com
2,2987002,0,86469,59.0,W,visa,debit,outlook.com
3,2987003,0,86499,50.0,W,mastercard,debit,yahoo.com
4,2987004,0,86506,50.0,H,mastercard,credit,gmail.com


In [69]:
df1.groupby("Product_Category")["TransactionID"].count()

Product_Category
C     68519
H     33024
R     37699
S     11628
W    439670
Name: TransactionID, dtype: int64

In [ ]:
fraud = df1[df1["isFraud"] == 1]
fraud.groupby("Product_Category")["TransactionID"].count()

KeyError: 'Card6'

In [83]:
df1.groupby("Product_Category")["TransactionAmt"].max()

Product_Category
C      712.896
H      500.000
R     1800.000
S     1550.000
W    31937.391
Name: TransactionAmt, dtype: float64

In [90]:
fraud = df1[df1["isFraud"] == 1]
fraud.groupby("Card_name")["TransactionID"].count()

Card_name
american express      239
discover              514
mastercard           6496
visa                13414
Name: TransactionID, dtype: int64

In [93]:
non_fraud = df1[df1["isFraud"] == 0]
non_fraud.groupby("card6")["TransactionID"].count()

card6
charge card            15
credit             139036
debit              430796
debit or credit        30
Name: TransactionID, dtype: int64

In [94]:
card_analysis = df1.groupby("card6", dropna=False).agg(
    total_transactions=("TransactionID", "count"),
    fraud_count=("isFraud", "sum"),
    avg_amount=("TransactionAmt", "mean")
)

card_analysis["fraud_rate"] = (
    df1.groupby("card6", dropna=False)["isFraud"].mean() * 100
)

print(card_analysis.sort_values("fraud_rate", ascending=False))

                 total_transactions  fraud_count  avg_amount  fraud_rate
card6                                                                   
credit                       148986         9950  190.034359    6.678480
debit                        441509        10713  116.469822    2.426451
charge card                      15            0   59.528133    0.000000
debit or credit                  30            0  104.065667    0.000000


In [ ]:
df1[df1["card6"].isin(["charge card", "debit or credit"])][
    ["card6", "isFraud", "TransactionAmt"]
]

,TransactionID,card6,isFraud,TransactionAmt
16408,3003408,debit or credit,0,58.510
23523,3010523,charge card,0,19.757
27859,3014859,debit or credit,0,25.000
43216,3030216,debit or credit,0,58.430
47201,3034201,charge card,0,28.509
47306,3034306,charge card,0,50.604
54633,3041633,debit or credit,0,512.610
61895,3048895,debit or credit,0,160.470
67872,3054872,debit or credit,0,108.430
81130,3068130,debit or credit,0,57.950


In [97]:
df2.duplicated().sum()

np.int64(0)

In [99]:
df1.to_csv("C:/Users/hp/Documents/Fraud-Detection-Project/DATA/Clean Data/train_transaction.csv")


In [100]:
df2.to_csv("C:/Users/hp/Documents/Fraud-Detection-Project/DATA/Clean Data/train_identity.csv")